# Quantum Circuit Smell Intelligence — Kaggle run

Runs `qcs_pipeline` (the rule-mining "smell detector" pipeline) directly against
the `veerukhannan/mnisq-optbench-pairs` dataset's pre-computed `(input_qasm, target_qasm)`
pairs — Qiskit's transpiler has already been run to produce these, so Step 1 here
just re-diffs the already-paired QASM instead of re-transpiling from scratch.

**Before running:**
1. Attach the dataset: **Add Input** (right sidebar) → search `mnisq-optbench-pairs` → Add.
2. Turn on **Internet**: Notebook settings (right sidebar) → Internet → On. Needed for `git clone` and `pip install`.
3. The source repo (`veerakrish/quantum-circuit-smell-intelligence`) is **private**, so add a GitHub
   Personal Access Token (scope: `repo`) as a Kaggle Secret: **Add-ons → Secrets → Add a new secret**,
   name it `GITHUB_TOKEN`, paste the token value. Then attach it to this notebook.


In [ ]:
# Confirm the dataset is attached and see its exact mounted folder name
import glob
print(glob.glob('/kaggle/input/*'))

In [ ]:
# Clone the private repo using a Kaggle Secret (never paste the token in plaintext in a cell)
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

REPO_DIR = "/kaggle/working/quantum-circuit-smell-intelligence"
!rm -rf {REPO_DIR}
!git clone --quiet https://{GITHUB_TOKEN}@github.com/veerakrish/quantum-circuit-smell-intelligence.git {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
# qcs_pipeline itself only needs Qiskit + PyZX (no torch/torch-geometric —
# those are only for the separate neural qco_pipeline). Kaggle's base image
# already has numpy/scipy/pandas/pyarrow.
!pip install -q qiskit qiskit-aer pyzx

In [ ]:
# Locate the pairs_k*.parquet chunk files inside the attached dataset
from pathlib import Path
from qcs_pipeline.mining.from_kaggle_pairs import find_pair_chunks

# Adjust this if the printed folder name in the first cell differs
DATASET_ROOT = Path("/kaggle/input/mnisq-optbench-pairs")

chunk_files = find_pair_chunks(DATASET_ROOT)
print(f"Found {len(chunk_files)} chunk files")
print(chunk_files[:5])

## Peek before committing

Check the actual `opt_level` distribution and what `fidelity_input`/`fidelity_target`
hold, per the caveats from the design discussion — don't assume, verify.

In [ ]:
import pandas as pd

df_peek = pd.read_parquet(chunk_files[0])
print(df_peek.columns.tolist())
print("\nopt_level value counts:")
print(df_peek["opt_level"].value_counts())
print("\nfidelity_input / fidelity_target sample:")
print(df_peek[["fidelity_input", "fidelity_target", "input_n_gates", "target_n_gates"]].head())

In [ ]:
# Step 1 — mine patterns from every chunk (fast path: reuses the already-
# transpiled target_qasm, no re-transpiling). Set opt_level_filter to whatever
# the cell above shows is the dominant/consistent level. Start with
# max_rows_per_chunk set low to sanity-check the run within a few minutes,
# then rerun with None for the full pass.
from qcs_pipeline.mining.from_kaggle_pairs import mine_from_parquet

mine_from_parquet(
    dataset_root=DATASET_ROOT,
    out_path=Path("/kaggle/working/mined_pairs.jsonl"),
    opt_level_filter=3,       # match phase0/horizontal_pairs.py's default optimization_level=3 — adjust per the cell above
    max_rows_per_chunk=500,   # sanity-check pass; set to None for the full dataset
)

In [ ]:
# Step 2 — canonicalize into a deduped, frequency-ranked rule database
from qcs_pipeline.rules.rule_database import build_rule_database

db = build_rule_database(Path("/kaggle/working/mined_pairs.jsonl"), min_frequency=2)
db.to_json(Path("/kaggle/working/rules.json"))

entries = db.entries()
print(f"{len(entries)} unique rules ({sum(e.conflict for e in entries)} flagged conflicting)")
for e in sorted(entries, key=lambda x: -x.frequency)[:10]:
    print(e.frequency, [op.name for op in e.pattern], "->", [op.name for op in e.rewrite])

In [ ]:
# Apply the smell detector + exact-fidelity verified repair to a sample circuit
from qcs_pipeline.pipeline import QuantumCircuitSmellOptimizer

optimizer = QuantumCircuitSmellOptimizer(rule_db_path=Path("/kaggle/working/rules.json"))

sample_raw_qasm = df_peek.iloc[0]["input_qasm"]
result = optimizer.optimize(sample_raw_qasm)

print(f"{result.n_gates_before} -> {result.n_gates_after} gates")
print("fidelity:", result.fidelity)
print("rule applications:", len(result.applied_smells))
print("quarantined this run:", result.quarantined_rule_count)

## Notes

- **Memory:** chunks are processed and released one at a time, so peak RAM stays
  around one parquet chunk (~150 MB) plus whatever a single circuit's exact-fidelity
  simulation needs — see the RAM discussion for how that scales with qubit count.
- **Session limits:** Kaggle notebook sessions are time-boxed (check the current
  published limit in Kaggle's docs — it has changed over time). For a full-dataset
  run, set `max_rows_per_chunk=None` and consider running it as a saved/scheduled
  Kaggle notebook version rather than interactively.
- **Outputs:** `/kaggle/working/` persists as the notebook's output — `mined_pairs.jsonl`
  and `rules.json` will be downloadable from the notebook's Output tab after a run.
- If `opt_level` turns out to be mixed rather than a single dominant value, rerun
  Step 1 once per level and compare the resulting rule databases rather than merging.